In [43]:
import math
import os
import scipy
from scipy.optimize import lsq_linear
import numpy as np
from scipy.linalg import null_space
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal, halfnorm
import random
from scipy.io import loadmat
import random
import pickle
import sys
from sklearn.linear_model import RidgeCV
sys.path.append(r"c:\Users\katie\OneDrive\Documents\GitHub\trial")
import PCA_Regress as pcar
from brokenaxes import brokenaxes
from matplotlib.gridspec import GridSpec

In [44]:
base_path =r"c:\Users\katie\OneDrive\Desktop\Thesis"
with open(base_path+'\J_neu.pkl', "rb") as input_file:
    J_pickle = pickle.load(input_file)
del input_file

file_path = os.path.join(base_path, 'N_neu.pkl')
with open(file_path, "rb") as input_file:
    N_pickle = pickle.load(input_file)
del input_file

base_path =r"c:\Users\katie\OneDrive\Desktop\Thesis"
with open(base_path+'\J_mus.pkl', "rb") as input_file:
    J_pickle_m = pickle.load(input_file)
del input_file

ile_path = os.path.join(base_path, 'N_mus.pkl')
with open(ile_path, "rb") as input_file:
    N_pickle_m = pickle.load(input_file)
del input_file

# base_path = "/Users/kb6113/Desktop/Thesis"
# with open(base_path+'/J_neu.pkl', "rb") as input_file:
#     J_pickle = pickle.load(input_file)
# del input_file

# with open(base_path+'/J_mus.pkl', "rb") as input_file:
#     J_pickle_m = pickle.load(input_file)
# del input_file

J_all_tensor = J_pickle['J_all']['interpPSTH']
J_M1_tensor = J_pickle['J_M1']['interpPSTH']
J_PMd_tensor = J_pickle['J_PMd']['interpPSTH']
J_idx = np.r_[0:18, 36:45]
J_ntm_tensor = J_all_tensor[J_idx, :, :]
J_mus_tensor = J_pickle_m['interpPSTH']


N_all_tensor = N_pickle['N_all']['interpPSTH']
N_M1_tensor = N_pickle['N_M1']['interpPSTH']
N_PMd_tensor = N_pickle['N_PMd']['interpPSTH']
N_mus_tensor = N_pickle_m['interpPSTH']

<>:2: SyntaxWarning: invalid escape sequence '\J'
<>:12: SyntaxWarning: invalid escape sequence '\J'
<>:2: SyntaxWarning: invalid escape sequence '\J'
<>:12: SyntaxWarning: invalid escape sequence '\J'
C:\Users\katie\AppData\Local\Temp\ipykernel_42856\3800015342.py:2: SyntaxWarning: invalid escape sequence '\J'
  with open(base_path+'\J_neu.pkl', "rb") as input_file:
C:\Users\katie\AppData\Local\Temp\ipykernel_42856\3800015342.py:12: SyntaxWarning: invalid escape sequence '\J'
  with open(base_path+'\J_mus.pkl', "rb") as input_file:


In [51]:
def red_rank (tensor_N, tensor_M, rank, pm = False): 
    """
    Running reduced rank regression 
    """

    if pm: 
    # splicing preparatory and movement bins and scaling frmo 0 to 1 and mean centering

        N_prep_start = 30
        N_prep_end = 81     # 81 because it will get spliced off otherwise

        # retrieving dataset specifications
        J, PMd = pcar.ident(tensor_N)

        # altering movement periods depending on dataset
        if J:
            N_move_start = 150
            N_move_end = 216
        else:
            N_move_start = 142
            N_move_end = 208
        
        N_idx = np.r_[N_prep_start:N_prep_end, N_move_start:N_move_end]
        N_cut = tensor_N[:,:, N_idx]

        if PMd:
            M_prep_start = N_prep_start
            M_prep_end = N_prep_end
            M_move_start = N_move_start
            M_move_end = N_move_end
        else:
            M_prep_start = N_prep_start + 5
            M_prep_end = N_prep_end + 5
            M_move_start = N_move_start + 5
            M_move_end = N_move_end + 5
        M_idx = np.r_[M_prep_start:M_prep_end, M_move_start:M_move_end]
        M_cut = tensor_M[:,:, M_idx]

        N_cut_scale = pcar.scaling(N_cut, tuning = False)
        M_cut_scale = pcar.scaling(M_cut, tuning = False)

        # finding W_ls 
        cov = N_cut_scale.T @ N_cut_scale
        W_LS = np.linalg.solve(cov, N_cut_scale.T @ M_cut_scale)
    
        # running PCA on Neu_move @ W
        pre_PCA = W_LS.T @ N_cut_scale.T @ N_cut_scale @ W_LS

    else: 
        Neu_pm, Neu_move, M_move = pcar.time_shift(tensor_N, tensor_M)

        # finding W_ls 
        cov = Neu_move.T @ Neu_move
        W_LS = np.linalg.solve(cov, Neu_move.T @ M_move)
        
        
        # running PCA on Neu_move @ W
        pre_PCA = W_LS.T @ Neu_move.T @ Neu_move @ W_LS

     
    U, S_, V_T = np.linalg.svd(pre_PCA)
    V_rank = V_T.T[:,:rank]
    
    W_RRR = W_LS @ V_rank @ V_rank.T

    return W_RRR, V_rank, W_LS



In [52]:
rank = 3
W_RRR, V_rank, W_LS = red_rank(J_ntm_tensor, J_mus_tensor, rank, pm = True)
cond = J_ntm_tensor.shape[0]
print(W_RRR.shape)
regress_N, Neu_move, regress_M = pcar.time_shift(J_ntm_tensor, J_mus_tensor)
U, S, V = np.linalg.svd(W_RRR)
M_hat = Neu_move @ W_RRR
print(regress_M.shape[1])
W_potent = U[:, :3]
other_null = null_space(U)
W_null = null_space(W_RRR.T)
W_null = W_null[:, :3]
print(W_null.shape)
print(W_potent.shape)
check = W_potent.T @ W_null
R2_total = 1 - np.sum((regress_M - M_hat)**2) / np.sum((regress_M - regress_M.mean(axis=0))**2)
print(np.sum(check, axis = 1))

(202, 32)
32
(202, 3)
(202, 3)
[ 4.94396191e-17 -2.22911967e-16  4.35415592e-16]


In [41]:
tensor_N = J_ntm_tensor
tensor_M = J_mus_tensor
dimensions = 3
# retrieving number of conditions
cond = tensor_N.shape[0]

# retrieving dataset specifications
J, PMd = pcar.ident(tensor_N)

# tensor_N = slice(tensor_N)

if PMd: 
    tensor_M = slice(tensor_M)

# scaling, mean centering, and involving only the time periods needed for regression (the movement)
regress_N, move_N, regress_M = pcar.time_shift(tensor_N, tensor_M)
time_ct = regress_M.shape [0]
time_ct_neu = regress_N.shape [0]

# retrieving data projected onto the first N_dim and M_dim PCs
N_tilde,N_PCs = pcar.run_PCA(regress_N, dimensions)
M_tilde,PCs = pcar.run_PCA(regress_M, int(dimensions/2))

# how many time bins are included in the movement period
time_bins = int(time_ct / cond)

# how many time bins are included in the preparatory and movement period
time_bins_pm = int(time_ct_neu / cond)

# difference in bins = prep bins
diff_bin = int((time_bins_pm - time_bins))

# removing prep bins adn reshaping for ridge
regress_N = pcar.shape_tensor(N_tilde, conditions = cond, time_bins = time_bins_pm)
N_tens_spliced = regress_N[:,:, diff_bin:]
regress_N_sp = pcar.shape_matrix(N_tens_spliced)

# running through ridge regression
W, R2_total, R2_dim, MSE_all, RMSE_all = pcar.r_regress(regress_N_sp, M_tilde, num_bins = time_bins, J = J, PMd = PMd, cv = True)
print(R2_total)

[14  5  1 21 22  4 11 13 16  9 19 26  7  6 23 15 10 24  8 18 12  3]
>>> best_lam returning: 20.30917620904739
0.6995472514952445


In [53]:
regress_N, move_N, regress_M = pcar.time_shift(tensor_N, tensor_M)

# lengths of conditions x time, regress M only has movement, whereas regress_N has prep and movement
time_ct = regress_M.shape [0]
time_ct_neu = regress_N.shape [0]

# how many time bins are included in the movement period for a single condition
time_bins = int(time_ct / cond)

# how many time bins are included in the preparatory and movement period per condition
time_bins_pm = int(time_ct_neu / cond)

# difference in bins = just prep bins ie where the movement period starts for each condition
diff_bin = int((time_bins_pm - time_bins))

# isolating the preparatory and movement bins
N_tilde_tens = pcar.shape_tensor(regress_N, cond, time_bins = time_bins_pm)
N_tilde_tens_move = N_tilde_tens[:,:,diff_bin:]
N_tilde_tens_prep = N_tilde_tens[:,:,:diff_bin]

# reshape into matrices for tuning computation
neu_move = pcar.shape_matrix(N_tilde_tens_move)
neu_prep = pcar.shape_matrix(N_tilde_tens_prep)

In [54]:
 # movement null and potent space for gamma
N_null_move = neu_move @ W_null
# N_nm_tensor = shape_tensor(N_null_move, cond)
N_null_move = N_null_move - N_null_move.mean(axis=0)     # the other one to comment out
# N_null_move = shape_matrix(N_nm_tensor)
null_move_frob = np.linalg.norm(N_null_move)**2
null_move_var = np.sum(np.var(N_null_move, axis=0))

N_pot_move = neu_move @ W_potent
# N_pm_tensor = shape_tensor(N_pot_move, cond)
N_pot_move = N_pot_move - N_pot_move.mean(axis=0)     # the one to comment out
# N_pot_move = shape_matrix(N_pm_tensor)
pot_move_frob = np.linalg.norm(N_pot_move)**2
pot_move_var = np.sum(np.var(N_pot_move, axis=0))

# computing gamma which is a scaling factor
gamma = null_move_var / pot_move_var
gamma2 = null_move_frob / pot_move_frob

# Null and potent projections of movement neural data
N_null_prep = neu_prep @ W_null
# N_np_tensor = shape_tensor(N_null_prep, cond)
N_null_prep = N_null_prep - N_null_prep.mean(axis=0) 
# N_null_prep = shape_matrix(N_np_tensor)       # subtract columns for variance
null_prep_frob = np.linalg.norm(N_null_prep)**2
null_prep_var = np.sum(np.var(N_null_prep, axis=0))

N_pot_prep = neu_prep @ W_potent
# N_pp_tensor = shape_tensor(N_pot_prep, cond)
N_pot_prep = N_pot_prep - N_pot_prep.mean(axis=0) 
# N_pot_prep = shape_matrix(N_pp_tensor)       # subtract columns for variance
pot_prep_frob = np.linalg.norm(N_pot_prep)**2
pot_prep_var = np.sum(np.var(N_pot_prep, axis=0))

# tuning ratio
var_tuning = (null_prep_var / pot_prep_var) * ( 1/ gamma )   # this is with using the sum of variance
frob_tuning = (null_prep_frob / pot_prep_frob) * (1 / gamma2 )   # this is with using the frobenius norm and not variance on the movement data

# fraction of prep in null space and potent space
null_fraction = null_prep_var / (null_prep_var + pot_prep_var)
pot_fraction  = pot_prep_var / (null_prep_var + pot_prep_var)

print("potent move variance: ", pot_move_var)
print("null move variance: ", null_move_var)
print("null prep variance:", null_prep_var)
print("potent prep variance: ", pot_prep_var)
print("Tuning with variance: ", var_tuning)
print("Tuning with frob: ", frob_tuning)
print("Prep fraction: ",  null_fraction)

potent move variance:  0.12757655154269554
null move variance:  0.05858897066592316
null prep variance: 0.15052118529254765
potent prep variance:  0.0013923054924743403
Tuning with variance:  235.40630136790452
Tuning with frob:  235.40630136790446
Prep fraction:  0.9908348792113226


In [5]:
tensor_N = J_ntm_tensor
tensor_M = J_mus_tensor

In [7]:
# Use all 108 neural conditions for PCA to get a better neural manifold
N_full_tensor = J_all_tensor  # all 108 conditions

# But for the regression N->M, use only the 27 matched conditions
N_matched_tensor = J_all_tensor[J_idx, :, :]  # 27 conditions matching muscle

# Run PCA on full 108-condition neural data
regress_N_full, _, M_split = pcar.time_shift(N_full_tensor, tensor_M)
N_tilde_full, PCs = pcar.run_PCA(regress_N_full, 6)
M_tilde, _ = pcar.run_PCA(M_split, 3)

# Project the 27 matched conditions onto those same PCs
regress_N_matched, N_move_matched, regress_M = pcar.time_shift(N_matched_tensor, tensor_M)
N_tilde_matched = regress_N_matched @ PCs


# fit W on movement only
cond = tensor_N.shape[0]
time_ct = regress_M.shape[0]
time_bins = int(time_ct / cond)
time_bins_pm = int(regress_N_matched.shape[0] / cond)
diff_bin = time_bins_pm - time_bins

N_tens = pcar.shape_tensor(N_tilde_matched, cond, time_bins_pm)
N_tilde_move = pcar.shape_matrix(N_tens[:,:,diff_bin:])
N_tilde_prep = pcar.shape_matrix(N_tens[:,:,:diff_bin])

# fit W on ALL movement data, no train/test split
C = N_tilde_matched.T @ N_tilde_matched
lam, _, _ = pcar.best_lam(N_tilde_matched, M_tilde, time_bins)
W = np.linalg.solve(C + lam * np.eye(6), N_tilde_matched.T @ M_tilde)

U, S, Vt = np.linalg.svd(W, full_matrices=True)
W_potent = U[:, :3]
W_null = U[:, 3:]

# check R2 of the regression
M_hat = N_tilde_matched @ W
R2 = 1 - np.sum((M_tilde - M_hat)**2) / np.sum((M_tilde - M_tilde.mean(axis=0))**2)
print("R2 of regression:", R2)

# check movement period potent vs null variance
N_pot_move = N_tilde_matched @ W_potent
N_null_move = N_tilde_matched @ W_null
N_pot_move -= N_pot_move.mean(axis=0)
N_null_move -= N_null_move.mean(axis=0)
print("Potent move norm^2:", np.linalg.norm(N_pot_move)**2)
print("Null move norm^2:  ", np.linalg.norm(N_null_move)**2)
print("Gamma:", np.linalg.norm(N_null_move)**2 / np.linalg.norm(N_pot_move)**2)

# check prep period
N_pot_prep = N_tilde_prep @ W_potent
N_null_prep = N_tilde_prep @ W_null
N_pot_prep -= N_pot_prep.mean(axis=0)
N_null_prep -= N_null_prep.mean(axis=0)
print("Potent prep norm^2:", np.linalg.norm(N_pot_prep)**2)
print("Null prep norm^2:  ", np.linalg.norm(N_null_prep)**2)
print("Raw prep null/pot ratio:", np.linalg.norm(N_null_prep)**2 / np.linalg.norm(N_pot_prep)**2)
print("Tuning ratio:", (np.linalg.norm(N_null_prep)**2 / np.linalg.norm(N_pot_prep)**2) / 
                       (np.linalg.norm(N_null_move)**2 / np.linalg.norm(N_pot_move)**2))

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 1369 is different from 2479)

In [8]:
W = np.linalg.solve(C + lam * np.eye(6), N_tilde_move.T @ M_tilde)
print("W shape:", W.shape)
print("W singular values:", np.linalg.svd(W, compute_uv=False))
U, S, Vt = np.linalg.svd(W, full_matrices=True)
print("U shape:", U.shape)

# verify potent captures more movement variance than null
N_pot = N_tilde_move @ U[:, :3]
N_nul = N_tilde_move @ U[:, 3:]
N_pot -= N_pot.mean(axis=0)
N_nul -= N_nul.mean(axis=0)
print("Move potent norm:", np.linalg.norm(N_pot)**2)
print("Move null norm:  ", np.linalg.norm(N_nul)**2)

# now check what M_tilde looks like — is it actually varying across conditions?
print("M_tilde variance per dim:", np.var(M_tilde, axis=0))
print("M_tilde total variance:", np.sum(np.var(M_tilde, axis=0)))

# and check N_tilde_prep variance
print("N_tilde_prep total variance:", np.sum(np.var(N_tilde_prep, axis=0)))
print("N_tilde_move total variance:", np.sum(np.var(N_tilde_move, axis=0)))

NameError: name 'C' is not defined